# Systemy uczące się - Zad. dom. 1: Minimalizacja ryzyka empirycznego
Celem zadania jest zaimplementowanie własnego drzewa decyzyjnego wykorzystującego idee minimalizacji ryzyka empirycznego. 

### Autor rozwiązania
Uzupełnij poniższe informacje umieszczając swoje imię i nazwisko oraz numer indeksu:

In [ ]:
NAME = "Jan Nowakowski"
ID = "155042"

## Twoja implementacja

Twoim celem jest uzupełnić poniższą klasę `TreeNode` tak by po wywołaniu `TreeNode.fit` tworzone było drzewo decyzyjne minimalizujące ryzyko empiryczne. Drzewo powinno wspierać problem klasyfikacji wieloklasowej (jak w przykładzie poniżej). Zaimplementowany algorytm nie musi (ale może) być analogiczny do zaprezentowanego na zajęciach algorytmu dla klasyfikacji. Wszelkie przejawy inwencji twórczej wskazane. **Pozostaw komenatrze w kodzie, które wyjaśniają Twoje rozwiązanie.**

Schemat oceniania:
- wynik na zbiorze Iris (automatyczna ewaluacja) celność klasyfikacji >= prostego baseline'u + 10%: +40%,
- wynik na ukrytym zbiorze testowym 1 (automatyczna ewaluacja) celność klasyfikacji >= prostego baseline'u + 15%: +30%,
- wynik na ukrytym zbiorze testowym 2 (automatyczna ewaluacja) celność klasyfikacji >= prostego baseline'u + 5%: +30%.

Niedozwolone jest korzystanie z zewnętrznych bibliotek do tworzenia drzewa decyzyjnego (np. scikit-learn). 
Możesz jedynie korzystać z biblioteki numpy.

#### Uwaga: Możesz dowolnie modyfikować elementy tego notebooka (wstawiać komórki i zmieniać kod), o ile będzie się w nim na koniec znajdowała kompletna implementacja klasy `TreeNode` w jednej komórce.

In [3]:
import numpy as np

class TreeNode:
    def __init__(self, max_depth: int = 10, min_samples_split: int = 2, current_depth: int = 0):
        self.left: TreeNode | None = None  # wierzchołek znajdujący się po lewej stronie
        self.right: TreeNode | None = None  # wierzchołek znajdujący się po prawej stronie
        
        # Parametry węzła decyzyjnego
        self.feature_index: int | None = None  # indeks cechy, po której następuje podział
        self.threshold: float | None = None    # wartość progowa podziału
        self.value: int | float | None = None  # przewidywana klasa (jeśli węzeł jest liściem)
        
        # Hiperparametry zapobiegające przeuczeniu (overfittingowi)
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.current_depth = current_depth

    def fit(self, data: np.ndarray, target: np.ndarray) -> None:
        """
        Buduje drzewo decyzyjne na podstawie danych treningowych.
        """
        num_samples, num_features = data.shape
        unique_classes = np.unique(target)

        # 1. Sprawdzenie warunków stopu
        # Zatrzymujemy podział, jeśli wszystkie próbki należą do jednej klasy, 
        # osiągnięto maksymalną głębokość drzewa lub jest za mało próbek by dokonać podziału.
        if (len(unique_classes) == 1 or 
            num_samples < self.min_samples_split or 
            self.current_depth >= self.max_depth):
            self.value = self._most_common_class(target)
            return

        best_gini = float('inf')
        best_split = None

        # 2. Poszukiwanie optymalnego podziału (Minimalizacja ryzyka empirycznego)
        # Iterujemy przez wszystkie cechy oraz ich unikalne wartości, szukając takiego punktu
        # podziału, który zminimalizuje ważony wskaźnik zanieczyszczenia Giniego.
        for feat_idx in range(num_features):
            thresholds = np.unique(data[:, feat_idx])
            
            for threshold in thresholds:
                # Dzielenie danych na dwie grupy na podstawie obecnego progu
                left_indices = data[:, feat_idx] <= threshold
                right_indices = data[:, feat_idx] > threshold

                # Ignoruj podziały, które nie rozdzielają danych (jedna strona jest pusta)
                if np.sum(left_indices) == 0 or np.sum(right_indices) == 0:
                    continue

                # Obliczenie wskaźnika Giniego dla obu podzbiorów
                gini_left = self._gini(target[left_indices])
                gini_right = self._gini(target[right_indices])

                # Obliczenie ważonego wskaźnika Giniego dla całego podziału
                weight_left = np.sum(left_indices) / num_samples
                weight_right = np.sum(right_indices) / num_samples
                gini_split = weight_left * gini_left + weight_right * gini_right

                # Zapisanie podziału, jeśli jest lepszy niż dotychczasowe
                if gini_split < best_gini:
                    best_gini = gini_split
                    best_split = {
                        'feature_index': feat_idx,
                        'threshold': threshold,
                        'left_indices': left_indices,
                        'right_indices': right_indices
                    }

        # 3. Rekurencyjne budowanie drzewa
        if best_split is not None:
            self.feature_index = best_split['feature_index']
            self.threshold = best_split['threshold']

            # Inicjalizacja dzieci ze zaktualizowaną głębokością
            self.left = TreeNode(self.max_depth, self.min_samples_split, self.current_depth + 1)
            self.right = TreeNode(self.max_depth, self.min_samples_split, self.current_depth + 1)

            # Trenowanie dzieci na odpowiednich podzbiorach
            self.left.fit(data[best_split['left_indices']], target[best_split['left_indices']])
            self.right.fit(data[best_split['right_indices']], target[best_split['right_indices']])
        else:
            # Jeśli nie znaleziono żadnego sensownego podziału, stwórz liść
            self.value = self._most_common_class(target)

    def predict(self, data: np.ndarray) -> np.ndarray:
        """
        Zwraca przewidywane klasy dla dostarczonej macierzy cech.
        """
        # Aplikuje funkcję _predict_single do każdego rzędu macierzy wejściowej
        return np.array([self._predict_single(x) for x in data])

    def _predict_single(self, x: np.ndarray):
        """
        Rekurencyjnie przechodzi przez drzewo dla pojedynczej próbki.
        """
        # Jeśli jesteśmy w liściu, zwracamy jego klasę
        if self.value is not None:
            return self.value
        
        # W przeciwnym razie idziemy w dół drzewa zgodnie z warunkiem podziału
        if x[self.feature_index] <= self.threshold:
            return self.left._predict_single(x)
        else:
            return self.right._predict_single(x)

    def _gini(self, y: np.ndarray) -> float:
        """
        Oblicza wskaźnik zanieczyszczenia Giniego (Gini Impurity) dla wektora etykiet.
        """
        m = len(y)
        if m == 0:
            return 0.0
        
        # Zliczenie wystąpień każdej z klas
        _, counts = np.unique(y, return_counts=True)
        # Obliczenie prawdopodobieństwa wystąpienia każdej z klas
        probabilities = counts / m
        # Wzór na Gini: 1 - suma kwadratów prawdopodobieństw
        return 1.0 - np.sum(probabilities ** 2)

    def _most_common_class(self, y: np.ndarray):
        """
        Zwraca klasę, która występuje najczęściej w danym wektorze.
        Przydatne podczas decydowania o wartości liścia.
        """
        classes, counts = np.unique(y, return_counts=True)
        return classes[np.argmax(counts)]

ModuleNotFoundError: No module named 'numpy'

## Przykład trenowanie i testowania drzewa
 
Później znajduje się przykład trenowania i testowania drzewa na zbiorze danych `iris`, który zawierający 150 próbek irysów, z czego każda próbka zawiera 4 atrybuty: długość i szerokość płatków oraz długość i szerokość działki kielicha. Każda próbka należy do jednej z trzech klas: `setosa`, `versicolor` lub `virginica`, które są zakodowane jak int.

Możesz go wykorzystać do testowania swojej implementacji. Możesz też zaimplementować własne testy lub użyć innych zbiorów danych, np. innych [zbiorów danych z scikit-learn](https://scikit-learn.org/stable/datasets/toy_dataset.html#toy-datasets).

In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.33, random_state=2024)

tree_model = TreeNode()
tree_model.fit(X_train, y_train)
y_pred = tree_model.predict(X_test)
print(accuracy_score(y_test, y_pred))

ModuleNotFoundError: No module named 'sklearn'